# TP2 — Análisis exploratorio (EDA)

**Proyecto:** predicción de fumadores (`smoking`). Ejecutar las celdas en orden.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_etiq = pd.read_csv("/content/drive/MyDrive/DiploIA/TP2/1-data/1-Raw/smoking_prediction.csv")


In [ ]:
df_etiq.head()

In [ ]:
sns.countplot(data=df_etiq, x="smoking")
plt.title("Distribución de smoking")
plt.xlabel("Smoking (0 = No, 1 = Sí)")
plt.ylabel("Cantidad de personas")
plt.show()

In [ ]:
sns.countplot(data=df_etiq, x="gender", hue="smoking")
plt.title("Smoking según género")
plt.xlabel("Género")
plt.ylabel("Cantidad de personas")
plt.legend(title="Smoking")
plt.show()

In [ ]:
pd.crosstab(df_etiq["gender"], df_etiq["smoking"],normalize="index") * 100

**El 55% de los hombres fuma, Vs las mujeres solo fuman el 4%.**

In [ ]:
plt.figure(figsize=(12, 5))
sns.countplot(data=df_etiq, x="age", hue="smoking")
plt.title("Smoking según edad")
plt.xlabel("Edad")
plt.ylabel("Cantidad de personas")
plt.show()

In [ ]:
pd.crosstab(df_etiq["age"],df_etiq["smoking"],normalize="index") * 100

**A medida que aumenta la edad, baja el % de fumadores**

In [ ]:
sns.countplot(data=df_etiq, x="tartar", hue="smoking")
plt.title("Smoking en personas con de sarro")
plt.xlabel("Sarro dental")
plt.ylabel("Cantidad de personas")
plt.show()

In [ ]:
pd.crosstab(df_etiq["tartar"],df_etiq["smoking"],normalize="index") * 100

In [ ]:
sns.countplot(data=df_etiq, x="dental caries", hue="smoking")
plt.title("Smoking según personas con caries")
plt.xlabel("con cariers")
plt.ylabel("Cantidad de personas")
plt.show()

In [ ]:
pd.crosstab(df_etiq["dental caries"],df_etiq["smoking"],normalize="index") * 100

**Las personas con sarro o caries presentan un mayor porcentaje de fumadores**

In [ ]:
VarFisicas = ["height(cm)", "weight(kg)", "waist(cm)"]
for columna in VarFisicas:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_etiq, x="smoking", y=columna)
    plt.title(f"{columna} según smoking")
    plt.show()

In [ ]:
df_etiq.groupby("smoking")[VarFisicas].median()

**Los fumadores presentan mayores valores de altura, peso y cintura, aunque hay que ver si esta mas relacionado al genero que al fumador no fumador**

In [ ]:
VarFisicas = ["height(cm)", "weight(kg)", "waist(cm)"]
for columna in VarFisicas:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_etiq, x="gender", y=columna, hue="smoking")
    plt.title(f"{columna} según género y smoking")
    plt.show()

In [ ]:
df_etiq.groupby(["gender", "smoking"])[VarFisicas].median()

**Al separar por género, las diferencias de altura y peso desaparecen**

In [ ]:
VarClinicas = ["systolic", "relaxation", "fasting blood sugar",
"Cholesterol", "triglyceride", "HDL", "LDL","hemoglobin", "serum creatinine", "AST", "ALT", "Gtp"]
df_etiq.groupby("smoking")[VarClinicas].median().T

In [ ]:
medianas = df_etiq.groupby("smoking")[VarClinicas].median().T
medianas["Diferencia %"] = ((medianas[1.0] - medianas[0.0]) / medianas[0.0]) * 100
medianas.round(2)

In [ ]:
VarClinicasDestacadas = ["Gtp", "triglyceride", "ALT", "HDL"]
for columna in VarClinicasDestacadas:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df_etiq, x="smoking", y=columna)
    plt.title(f"{columna} según smoking")
    plt.show()

In [ ]:
df_etiq.groupby(["gender", "smoking"])[VarClinicasDestacadas].median()

**Gtp y triglyceride son las variables que más se destacan, incluso después de considerar el género**

In [ ]:
df_etiq[VarClinicasDestacadas].describe().T[
    ["min", "25%", "50%", "75%", "max"]
]

**Valores maximos muy alejados de la mediana**

In [ ]:
for variable in VarClinicasDestacadas:
    Q1 = df_etiq[variable].quantile(0.25)
    Q3 = df_etiq[variable].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    extremos = df_etiq[(df_etiq[variable] < limite_inferior) |
        (df_etiq[variable] > limite_superior)]
    print(variable, ":", len(extremos),
          "valores extremos (",round(len(extremos) / len(df_etiq) * 100, 2), "%)")

**valores extremos en las cuatro variables, principalmente en Gtp (8,66%) y ALT (6,70%)**

In [ ]:
excluir = ["ID", "smoking", "dental caries","hearing(left)", "hearing(right)", "Urine protein"]
correlaciones = (df_etiq.drop(columns=excluir).select_dtypes(include="number").corr())
plt.figure(figsize=(16, 10))
sns.heatmap(correlaciones, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlación entre variables numéricas")
plt.show()

**Correlaciones altas entre peso y cintura (0,82), presión sistólica y diastólica (0,76), colesterol y LDL (0,74), y AST y ALT (0,74).**
Cintura y peso tienen una corelacion bastante alta, evaluar durante el entrenamiento si conviene eliminar alguna

In [ ]:
df_etiq.groupby(["gender", "smoking"])[VarClinicas].median().T

In [ ]:
for genero in ["F", "M"]:
    datos = df_etiq[df_etiq["gender"] == genero]
    medianas = datos.groupby("smoking")[VarClinicas].median().T
    medianas["Diferencia %"] = ((medianas[1.0] - medianas[0.0]) / medianas[0.0]) * 100
    print("Género:", genero)
    display(medianas.round(2))

In [ ]:
for variable in VarClinicasDestacadas:
    plt.figure(figsize=(9, 4))

    sns.boxplot(
        data=df_etiq,
        x="gender",
        y=variable,
        hue="smoking",
        showfliers=False
    )

    plt.title(f"{variable} según género y smoking")
    plt.show()

In [ ]:
tabla_edad_genero = pd.crosstab(
    [df_etiq["gender"], df_etiq["age"]],
    df_etiq["smoking"],
    normalize="index"
) * 100

tabla_edad_genero.round(2)

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_etiq,
    x="age",
    y="smoking",
    hue="gender",
    estimator="mean",
    errorbar=None,
    marker="o")
plt.title("Porcentaje de fumadores según edad y género")
plt.xlabel("Edad")
plt.ylabel("Proporción de fumadores")
plt.show()

In [ ]:
pd.crosstab(df_etiq["age"],df_etiq["gender"])

In [ ]:
df_40_60 = df_etiq[
    df_etiq["age"].between(40, 60)
]

df_40_60.groupby(
    ["gender", "smoking"]
)[["Gtp", "triglyceride"]].median()

**Entre los 40 y 60 años, Gtp y triglyceride siguen siendo mayores en fumadores de ambos géneros**

In [ ]:
df_40_60.groupby(
    ["gender", "age", "smoking"]
)[["Gtp", "triglyceride"]].median()

In [ ]:
df_40_60.groupby(
    ["gender", "age", "smoking"]
).size()

**Al comparar por género y edad (40 a 60 años), Gtp y triglyceride siguen siendo mayores en fumadores en todos los grupo**

##SALUD DENTAL

In [ ]:
pd.crosstab(
    [df_etiq["gender"], df_etiq["tartar"]],
    df_etiq["smoking"],
    normalize="index"
) * 100

In [ ]:
pd.crosstab(
    [df_etiq["gender"], df_etiq["dental caries"]],
    df_etiq["smoking"],
    normalize="index"
) * 100

**Al separar por género, el porcentaje de fumadores sigue siendo mayor entre quienes tienen sarro o caries. La diferencia es más marcada en hombres**

In [ ]:
df_etiq.nunique().sort_values().head(10)

In [ ]:
df_etiq["smoking"].value_counts(normalize=True) * 100

## Conclusiones del EDA

- **Distribución de fumadores:** el 36,66% fumadores Vs 63,34% no fumadores.
- **Género:** Diferencia entre hombres y mujeres. El 55,22% de los hombres fuma, frente al 4,28% de las mujeres.
- **Edad:** en los hombres, el porcentaje de fumadores es mayor entre los 40 y 50 años y disminuye a partir de los 55. En las mujeres también tiende a bajar, aunque hay muy pocos registros menores de 40 años.
- **Variables físicas:** inicialmente encontramos diferencias en altura y peso, pero al separar por género estas prácticamente desaparecen en las medianas.
- **Variables clínicas:** Gtp y triglyceride son las que más se destacan. Sus valores son mayores en fumadores, incluso al comparar por género y edad entre los 40 y 60 años.
- **Salud dental:** Mayor porcentaje de fumadores entre quienes presentan sarro o caries, tanto en hombres como en mujeres.
- **Correlaciones:** Variables presentan correlaciones altas: peso y cintura. Se decide conservarlas para evaluar su utilidad en los modelos.
- **Valores extremos:** Valores atípicos en algunas variables clínicas, pero no vamos a eliminarlos sin comprobar primero su impacto.
- **Variables a eliminar: oral** tiene un único valor e **ID** funciona como identificador, por lo que no los utilizaremos para entrenar.

En general, encontramos que **gender**, **age**, **Gtp**, **triglyceride**, **tartar** y **dental caries** podrían aportar información para predecir **smoking**.

**Se van a conservar el resto de las variables y evaluar su utilidad durante el entrenamiento**